# ID stability

> Tests that `stamp_notebook_ids` is deterministic, idempotent, preserves existing ids, and that `reconcile_ids_from_py` recovers ids from a generated `.py`.

In [ ]:
#| hide
import tempfile, json, os
from nbdev_mcp.utils.nbformat import stamp_notebook_ids, reconcile_ids_from_py

def _nb(cells): return {"cells": cells, "metadata": {}, "nbformat": 4, "nbformat_minor": 5}
def _code(src, cid=None):
    c = {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": src}
    if cid: c["id"] = cid
    return c

# ids hash the notebook BASENAME, so determinism holds for the SAME basename across dirs/machines
with tempfile.TemporaryDirectory() as d, tempfile.TemporaryDirectory() as d2:
    p, p2 = os.path.join(d, 't.ipynb'), os.path.join(d2, 't.ipynb')   # same basename
    json.dump(_nb([_code("#| export\nx=1"), _code("#| export\ny=2")]), open(p, 'w'))
    json.dump(_nb([_code("#| export\nx=1"), _code("#| export\ny=2")]), open(p2, 'w'))
    assert stamp_notebook_ids(p) == 2
    ids1 = [c['id'] for c in json.load(open(p))['cells']]
    assert all(ids1)
    assert stamp_notebook_ids(p) == 0                                   # idempotent
    stamp_notebook_ids(p2)
    assert [c['id'] for c in json.load(open(p2))['cells']] == ids1      # deterministic (same basename)
    p3 = os.path.join(d, 't3.ipynb')
    json.dump(_nb([_code("#| export\nx=1", "keep-me")]), open(p3, 'w'))
    stamp_notebook_ids(p3)
    assert json.load(open(p3))['cells'][0]['id'] == "keep-me"           # preserve existing

# reconcile ids from a generated .py's `# %% path #id` markers
with tempfile.TemporaryDirectory() as d:
    nbp, pyp = os.path.join(d, 'r.ipynb'), os.path.join(d, 'r.py')
    json.dump(_nb([_code("#| export\nfrom __future__ import annotations"), _code("#| export\ndef f(): pass")]), open(nbp, 'w'))
    open(pyp, 'w').write("# %% r.ipynb #aaaa1111\nfrom __future__ import annotations\n\n# %% r.ipynb #bbbb2222\ndef f(): pass\n")
    reconcile_ids_from_py(nbp, pyp)
    assert [c.get('id') for c in json.load(open(nbp))['cells']] == ["aaaa1111", "bbbb2222"]

print("id stability tests passed")